# R4c — 고친 학습 정답으로 큰 판정기(DeBERTa-v3-base)를 학습하면 문장 끝 판정을 넘는가

## 왜 하나

- R1c: 원래 정답에서는 판정기를 0.08B → 0.18B로 키워도 4토큰 엄격 재현율이 +0.027에 그쳤다.
- R4: 보정 정답에서는 위약을 빼면 판정기가 클수록 순이득이 컸다(+0.02 → +0.07). **정답 잡음이 크기 효과를 가렸을** 가능성.
- R4b: 학습 정답까지 고치자 작은 판정기(MiniLM)의 4토큰 재현율이 0.519 → 0.588로 올라 B 문장 끝(0.621)에 근접했다.

남은 질문: **고친 학습 정답 + 큰 판정기**면 4토큰 앞부분 판정이 문장 끝 판정을 따라잡거나 넘는가.

## 무엇을 하나

- 학습 방식은 **R1c base와 같다**(`cross-encoder/nli-deberta-v3-base`, 학습률 2e-5, 1에폭, 지점별 이진 분류, 배치 64·작은 배치 16). 학습 정답만 **R4b와 같이 보정**(드라이브 `r4b/labels_train.json`, Sonnet 재판정, API 추가 비용 없음).
- 체크포인트 선택은 R4b와 같이 cal-sub의 보정 정답(C1). 시드 42·7·2026.
- 비교: R1c base(같은 백본, 원래 학습 정답), R4b V0b(MiniLM, 같은 보정 학습 정답), B 문장 끝.
- 판정 방식 4토큰 고정(R1c·R4b와 같음). 평가는 dev-L 원래 정답·C1 둘 다.

## 사전 정의 기준 (결과 보기 전, SSOT §6에 고정)

- **R4c-1 보정 정답에서의 크기 효과(주)**: C1 4토큰 **base(보정 학습) 시드 평균 − R4b V0b 시드 평균 ≥ +0.03**.
- **R4c-2 목표**: C1 4토큰 base(보정 학습) 시드 평균 ≥ **B 문장 끝 C1 재현율(0.621)**. (보고: 시드별 B 문장 끝 대비 쌍체 부트스트랩)
- **R4c-3 비열등(원래 정답)**: 원래 정답 4토큰 base(보정 학습) − R1c base(원래 학습) 시드 평균 ≥ −0.02.
- 보고: 학습 정답 효과의 재현(C1에서 base 보정 − R1c base), 문장 끝 판정, 지연, 서비스 10%, AUROC, 과제별.
- 시드 3개가 모두 끝나야 판정한다. 일부만 끝나면 보고만 한다.
- 해석 규칙: R4c-1 충족이면 "정답이 깨끗해야 판정기 크기가 효과를 낸다". R4c-2 충족이면 "4토큰 앞부분 판정이 문장 끝 판정과 같거나 높은 재현율을 더 빨리" → 게이트웨이 교체 1순위(판정 1회 약 30ms, A100).

> **런타임: A100** (L4 가능, T4 비권장). R1c base 실측 시드당 80~145분 → **3시드 약 4~7시간**. 끝난 시드는 건너뛰므로 끊겨도 다시 실행하면 이어서 한다.
> **드라이브 용량**: 시드 42 가중치(약 740MB)만 남기고 나머지는 점수만 저장한다. **실행 전에 드라이브 휴지통을 비울 것.**
> 드라이브에 `student_vark`, `r0/`, `r1b/`, `r1c/`(base·mini scores.npy), `r4/r4_corrected_labels.json`, `r4b/labels_train.json`·`V0b_s*/scores.npy`가 있어야 한다.


## 1. 환경 준비

In [ ]:
!pip -q install pysbd sentence-transformers tiktoken
!git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || echo 'RAGTruth already cloned'

import json, time, random, os, gc, copy
import numpy as np, pandas as pd, torch, pysbd, tiktoken
from sklearn.metrics import roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

USE_DRIVE = True
OUT_DIR = '/content/drive/MyDrive/selfproject' if USE_DRIVE else '.'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
STUDENT_DIR = os.path.join(OUT_DIR, 'student_vark')
R0_DIR = os.path.join(OUT_DIR, 'r0')
R1_DIR = os.path.join(OUT_DIR, 'r4c')
R1B_DIR, R1C_DIR, R4_DIR, R4B_DIR = (os.path.join(OUT_DIR, d) for d in ['r1b', 'r1c', 'r4', 'r4b'])
os.makedirs(R1_DIR, exist_ok=True)
for p in [os.path.join(R4B_DIR, 'labels_train.json'), os.path.join(R4_DIR, 'r4_corrected_labels.json')]:
    assert os.path.exists(p), f'{p} 없음'
print('device', DEVICE)

## 2. 데이터 (R1b와 같음)

In [ ]:
BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources   = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{BASE}/source_info.jsonl')}
for r in responses:
    r['task_type'] = sources[r['source_id']]['task_type']

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))

TASKS = ['QA', 'Summary', 'Data2txt']
train_pool = [r for r in responses if r['split'] == 'train']          # test 분할은 사용하지 않음
random.seed(SEED)
dev = []
for tt in TASKS:
    dev += random.sample([r for r in train_pool if r['task_type'] == tt], 120)
dev_ids = {r['id'] for r in dev}
rest = [r for r in train_pool if r['id'] not in dev_ids]
rng = random.Random(11)
teacher_resp = []
for tt in TASKS:
    teacher_resp += rng.sample([r for r in rest if r['task_type'] == tt], 200)
rng.shuffle(teacher_resp)
val = teacher_resp[:int(len(teacher_resp) * 0.15)]                   # student 학습에 쓰지 않은 검증 분할
assert not (dev_ids & {r['id'] for r in val})
print(f'dev {len(dev)} 응답 | 검증 {len(val)} 응답')

In [ ]:
# 학습 응답: S2/S2-b의 gold-large와 같은 구성 (teacher 학습 510 + 추가 2,490). dev·검증과 겹치지 않음을 확인한다.
teacher_train = teacher_resp[len(val):]
used = dev_ids | {r['id'] for r in teacher_resp}
gl_extra = rng.sample([r for r in rest if r['id'] not in used], 3000 - len(teacher_train))
train = teacher_train + gl_extra
tr_ids = {r['id'] for r in train}
assert not (tr_ids & dev_ids) and not (tr_ids & {r['id'] for r in val}), '학습과 dev/검증 겹침'
print(f'학습 {len(train)} 응답 (dev·검증과 겹침 없음)')

In [ ]:
ENC = tiktoken.get_encoding('cl100k_base')
seg = pysbd.Segmenter(language='en', clean=False, char_span=True)

def tokenize_offsets(text):
    ids = ENC.encode(text)
    dec, starts = ENC.decode_with_offsets(ids)
    if dec != text:                       # 드문 경우: 멀티바이트 문자 경계 — 누적 디코드로 대체
        starts, pos = [], 0
        for i in range(len(ids)):
            starts.append(pos); pos = len(ENC.decode(ids[:i + 1]))
    ends = starts[1:] + [len(text)]
    return starts, ends

def prep(r):
    text = r['response']
    ts, te = tokenize_offsets(text)
    # 토큰의 '내용 시작' 위치: cl100k 토큰은 앞 공백을 포함하므로(" The"), 공백을 건너뛴 위치로 문장·환각 구간에 배정한다.
    # (v1에서는 공백 위치로 배정해 문장 경계 토큰이 앞 문장에 붙는 오류가 발생 — 발생 위치의 18%)
    cs = [a + (len(text[a:b]) - len(text[a:b].lstrip())) for a, b in zip(ts, te)]
    cs = [c if c < b else a for c, a, b in zip(cs, ts, te)]          # 공백뿐인 토큰
    spans = [(l['start'], l['end']) for l in (r.get('labels') or [])]
    bad = [j for j in range(len(ts)) if any(min(te[j], b) > max(cs[j], a) for a, b in spans)]
    sents = []
    for sp in seg.segment(text):
        if not sp.sent.strip(): continue
        a, b = sp.start, sp.end
        tok = [j for j in range(len(ts)) if a <= cs[j] < b]
        if not tok: continue
        y = int(any(min(b, e) > max(a, s) for s, e in spans))
        sents.append({'a': a, 'b': b, 't0': tok[0], 't1': tok[-1], 'y': y})
    sent_of = {t: sn['t0'] for sn in sents for t in range(sn['t0'], sn['t1'] + 1)}
    return {'id': str(r['id']), 'task_type': r['task_type'], 'gen_model': r['model'],
            'ref_key': r['source_id'], 'text': text, 'ts': ts, 'te': te, 'n_tok': len(ts),
            'onset': bad[0] if bad else None, 'sents': sents, 'sent_of': sent_of,
            'ends': {sn['t1'] for sn in sents}}

REFS = {}
def to_items(rs):
    out = []
    for r in rs:
        REFS.setdefault(r['source_id'], build_reference(sources[r['source_id']]))
        out.append(prep(r))
    return out

dev_items, val_items = to_items(dev), to_items(val)
for name, it in [('dev', dev_items), ('검증', val_items)]:
    n_h = sum(x['onset'] is not None for x in it)
    print(f'{name}: 응답 {len(it)} (환각 {n_h}, 없음 {len(it)-n_h}) | 토큰 {sum(x["n_tok"] for x in it):,} | '
          f'환각 없는 응답 토큰 {sum(x["n_tok"] for x in it if x["onset"] is None):,}')

## 3. R4b 보정 학습 정답 불러오기 (Sonnet 재판정, 추가 비용 없음)

In [ ]:
TRAIN_ITEMS = to_items(train)
ITEMS = {f'tr-{x["id"]}': x for x in TRAIN_ITEMS}
LABELS = {k: {int(i): v for i, v in d.items()} for k, d in json.load(open(os.path.join(R4B_DIR, 'labels_train.json'))).items()}
print(f'학습 재판정 {len(LABELS):,}/{len(ITEMS):,}')

In [ ]:
def content_starts(x):
    t = x['text']
    cs = [a + (len(t[a:b]) - len(t[a:b].lstrip())) for a, b in zip(x['ts'], x['te'])]
    return [c if c < b else a for c, a, b in zip(cs, x['ts'], x['te'])]

def locate(x, sn, phrase, cs):
    seg = x['text'][sn['a']:sn['b']]
    ph = (phrase or '').strip()
    k = seg.find(ph) if ph else -1
    if k < 0 and ph: k = seg.lower().find(ph.lower())
    if k < 0: return sn['t0']
    a, b = sn['a'] + k, sn['a'] + k + len(ph)
    for t in range(sn['t0'], sn['t1'] + 1):
        if min(x['te'][t], b) > max(cs[t], a): return t
    return sn['t0']

ADD_ONSET, flip, earlier = {}, 0, 0
for cid, x in ITEMS.items():
    labs = LABELS.get(cid)
    if not labs: continue
    cs, add = None, []
    for i, sn in enumerate(x['sents']):
        if sn['y'] == 0 and labs[i + 1]['label'] == 'H':
            cs = cs or content_starts(x)
            t = locate(x, sn, labs[i + 1]['phrase'], cs); ADD_ONSET[(x['id'], i)] = t; add.append(t)
    if add:
        if x['onset'] is None: flip += 1
        elif min(add) < x['onset']: earlier += 1
n_clean = sum(x['onset'] is None for x in TRAIN_ITEMS); n_hall = len(TRAIN_ITEMS) - n_clean
print(f'추가된 환각 문장 {len(ADD_ONSET):,} | 멀쩡 → 환각 {flip}/{n_clean} ({flip / max(n_clean, 1):.0%}) | 더 이른 환각 {earlier}/{n_hall} ({earlier / max(n_hall, 1):.0%})')

## 4. 학습 판정 지점 — 보정 학습 정답 (R4b와 같음)

In [ ]:
def points(item, s):
    pts = []
    for sn in item['sents']:
        for t in range(sn['t0'], sn['t1'] + 1):
            k = t - sn['t0'] + 1
            if (s is not None and k % s == 0) or t == sn['t1']:
                pts.append((t, item['text'][sn['a']:min(item['te'][t], sn['b'])].strip()))
    return pts


WIN_TOK = 10
train_items = TRAIN_ITEMS
by_tr = {str(r['id']): r for r in train}
TR, P_KEY, P_TXT, P_CAT = [], [], [], []
for x in train_items:
    text = x['text']
    spans = [(l['start'], l['end']) for l in (by_tr[x['id']].get('labels') or [])]
    cs = [a + (len(text[a:b]) - len(text[a:b].lstrip())) if text[a:b].strip() else a for a, b in zip(x['ts'], x['te'])]
    def s_onset(sn):                      # E2와 같다: 이 문장 안에서 환각 구간과 처음 겹치는 토큰
        for t in range(sn['t0'], sn['t1'] + 1):
            if any(min(x['te'][t], b) > max(cs[t], a) for a, b in spans): return t
        return None
    sid = {sn['t0']: i for i, sn in enumerate(x['sents'])}
    son = [s_onset(sn) for sn in x['sents']]
    son = [o if o is not None else ADD_ONSET.get((x['id'], i)) for i, o in enumerate(son)]   # R4b 보정 학습 정답
    groups = {}
    for t, txt in points(x, 2):
        if not txt: continue
        si = sid[x['sent_of'][t]]; o = son[si]
        if o is None: cat = 'neg'
        elif t < o: cat = 'pre'
        elif t <= o + WIN_TOK: cat = 'win'
        else: cat = 'post'
        g = len(P_TXT); P_KEY.append(x['ref_key']); P_TXT.append(txt); P_CAT.append(cat)
        if cat in ('win', 'post'): groups.setdefault(si, []).append(g)
    gl = []
    for si, lst in groups.items():
        win = [g for g in lst if P_CAT[g] == 'win']
        if not win:                       # 창 안에 지점이 없으면 시작 이후 첫 지점을 창으로
            P_CAT[lst[0]] = 'win'; win = [lst[0]]
        gl.append(win)
    TR.append({'groups': gl})
P_CAT = np.array(P_CAT)
print(f'학습 응답 {len(TR)} | 판정 지점 {len(P_TXT):,} | 환각 문장(다중 사례 단위) {sum(len(r["groups"]) for r in TR):,} |',
      pd.Series(P_CAT).value_counts().to_dict())
assert sum(len(r['groups']) for r in TR) > 0

## 5. 근거 검색과 평가용 보정 정답(R4 C1)

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

retriever = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)
K, MAX_LEN, CHUNK_WORDS, BS = 8, 512, 40, 32

chunk_cache = {}
def get_chunks(key):
    if key not in chunk_cache:
        w = REFS[key].split()
        ch = [' '.join(w[i:i + CHUNK_WORDS]) for i in range(0, max(len(w), 1), CHUNK_WORDS)] or [REFS[key]]
        chunk_cache[key] = (ch, retriever.encode(ch, convert_to_tensor=True, normalize_embeddings=True,
                                                 batch_size=128, show_progress_bar=False))
    return chunk_cache[key]

def contexts(keys, texts, bs=256):
    # 텍스트마다 근거 top-K 조각을 원문 순서로 이어 붙인 문자열
    out = []
    for i in range(0, len(texts), bs):
        embs = retriever.encode(texts[i:i + bs], convert_to_tensor=True, normalize_embeddings=True,
                                batch_size=bs, show_progress_bar=False)
        for key, e in zip(keys[i:i + bs], embs):
            ch, ce = get_chunks(key)
            top = torch.topk(ce @ e, min(K, ce.shape[0])).indices.tolist()
            out.append(' ... '.join(ch[t] for t in sorted(top)))
    return out


# R0 평가 세트(cal-L, dev-L)와 기존 판정기 점수
import pickle
R0P = pickle.load(open(os.path.join(R0_DIR, 'r0_items.pkl'), 'rb'))
Z0 = dict(np.load(os.path.join(R0_DIR, 'r0_scores.npz')))
CAL, DEV = R0P['items']['calL'], R0P['items']['devL']
Q_KEYS, Q_TEXTS = R0P['q_keys'], R0P['q_texts']
for k in set(Q_KEYS):
    REFS.setdefault(k, build_reference(sources[k]))

@torch.no_grad()
def topk_idx(keys, texts, bs=512):
    out = []
    for i in range(0, len(texts), bs):
        embs = retriever.encode(texts[i:i + bs], convert_to_tensor=True, normalize_embeddings=True,
                                batch_size=bs, show_progress_bar=False)
        for key, e in zip(keys[i:i + bs], embs):
            ch, ce = get_chunks(key)
            out.append(sorted(torch.topk(ce @ e, min(K, ce.shape[0])).indices.tolist()))
    return out
ctx_of = lambda key, idx: ' ... '.join(get_chunks(key)[0][j] for j in idx)

t0 = time.perf_counter()
P_IDX = topk_idx(P_KEY, P_TXT)
print(f'학습 지점 근거 검색 {time.perf_counter()-t0:.0f}초')

# 평가 질의: cal-L·dev-L 지점만 (R0와 같은 질의 번호)
EVAL_Q = sorted({q for x in CAL + DEV for _, q in x['pts2']})
t0 = time.perf_counter()
EQ_IDX = dict(zip(EVAL_Q, topk_idx([Q_KEYS[q] for q in EVAL_Q], [Q_TEXTS[q] for q in EVAL_Q])))
print(f'평가 질의 {len(EVAL_Q):,}개 근거 검색 {time.perf_counter()-t0:.0f}초')

# 체크포인트 선택용 cal-sub (cal-L에서 300응답 고정)
CALSUB = random.Random(1).sample(CAL, min(300, len(CAL)))

In [ ]:
# R4 보정 정답 C1 (Opus 재판정)을 cal-L·dev-L에 적용한 사본
CORR = json.load(open(os.path.join(R4_DIR, 'r4_corrected_labels.json')))['C1']
def apply_corr(items, split):
    out = []
    for x in items:
        c = CORR[split].get(x['id'])
        y = copy.copy(x); y['sents'] = [dict(sn) for sn in x['sents']]
        if c is not None:
            y['onset'] = c['onset']
            for i in c['added']: y['sents'][i - 1]['y'] = 1
        out.append(y)
    return out
CAL_C, DEV_C = apply_corr(CAL, 'calL'), apply_corr(DEV, 'devL')
LAB_SETS = {'orig': (CAL, DEV), 'C1': (CAL_C, DEV_C)}
for k, (c, d) in LAB_SETS.items():
    print(f"{k}: cal-L 환각 {sum(x['onset'] is not None for x in c)}, dev-L 환각 {sum(x['onset'] is not None for x in d)}")
# 체크포인트 선택은 보정 정답 기준 cal-sub (같은 300응답)
_byid = {x['id']: x for x in CAL_C}
CALSUB = [_byid[x['id']] for x in CALSUB]

## 6. 평가 함수 (R1b와 같음)

In [ ]:
SC = dict(Z0)

def series(x, scorer):
    m, k = scorer.split('|')
    S_ = SC[m]
    if k == 'end':
        return [(t, S_[q]) for t, q in x['pts2'] if t in x['ends']]
    if k == 'ew2':
        out, s = [], None
        for t, q in x['pts2']:
            v = float(S_[q]); s = v if s is None else 0.5 * v + 0.5 * s
            out.append((t, s))
        return out
    s = int(k)
    return [(t, S_[q]) for t, q in x['pts2'] if t in x['ends'] or (t - x['sent_of'][t] + 1) % s == 0]

In [ ]:
def calibrate(items, scorer, arl0):
    clean = [x for x in items if x['onset'] is None]
    n_tok = sum(x['n_tok'] for x in clean)
    sc = np.sort(np.concatenate([[s for _, s in series(x, scorer)] for x in clean]))[::-1]
    n_alarm = max(1, int(n_tok / arl0))
    return float(sc[n_alarm - 1]) + 1e-6 if n_alarm <= len(sc) else float(sc[-1])

def evaluate(items, scorer, thr):
    # 엄격 기준(주 지표, 선행 연구와 같은 정의): 응답의 **첫 경보**가 발생 θ 이후여야 탐지로 인정.
    #   θ 전에 경보가 울리면 그 응답은 오경보로 끝난 것(미탐지)으로 본다.
    # 관대 기준(참고): θ 전 경보는 무시하고 θ 이후 첫 경보를 탐지로 인정 — v1의 주 지표였으나 재현율이 부풀려짐.
    clean = [x for x in items if x['onset'] is None]
    hall = [x for x in items if x['onset'] is not None]
    alarms = sum(sum(s >= thr for _, s in series(x, scorer)) for x in clean)
    arl = sum(x['n_tok'] for x in clean) / max(alarms, 1)
    clean_flag = np.mean([any(s >= thr for _, s in series(x, scorer)) for x in clean])
    st_det, st_d, st_ch, cedd, le_det, pre = 0, [], [], [], 0, 0
    for x in hall:
        ser = series(x, scorer)
        first = next((t for t, s in ser if s >= thr), None)
        le_hit = next((t for t, s in ser if t >= x['onset'] and s >= thr), None)
        le_det += le_hit is not None
        pre += first is not None and first < x['onset']
        if first is not None and first >= x['onset']:
            st_det += 1; st_d.append(first - x['onset']); cedd.append(first - x['onset'])
            st_ch.append(x['te'][first] - x['ts'][x['onset']])
        else:
            cedd.append(x['n_tok'] - x['onset'])
    return {'realized_ARL0_dev': arl, 'clean_resp_flagged': clean_flag,
            'recall': st_det / len(hall), 'delay_tok_mean': np.mean(st_d) if st_d else np.nan,
            'delay_tok_median': np.median(st_d) if st_d else np.nan,
            'delay_chars_median': np.median(st_ch) if st_ch else np.nan,
            'censored_EDD_tok': np.mean(cedd), 'pre_onset_alarm_rate': pre / len(hall),
            'recall_lenient': le_det / len(hall),
            'calls_per_resp': np.mean([len(series(x, scorer)) for x in items])}

def calibrate_flag(items, scorer, rate):
    # 서비스 기준 보정: 환각 없는 응답 중 표시되는 비율이 rate 이하가 되는 임계값 (응답별 최대 점수의 분위수)
    mx = np.array([max([s for _, s in series(x, scorer)] or [-1e9]) for x in items if x['onset'] is None])
    return float(np.quantile(mx, 1 - rate)) + 1e-6

In [ ]:
def resp_auc(items, m):
    y = [int(x['onset'] is not None) for x in items]
    s = [max([SC[m][q] for _, q in x['pts2']] or [-1e9]) for x in items]
    return roc_auc_score(y, s)

def sent_auc(items, m):
    ys, ss = [], []
    for x in items:
        d = dict(x['pts2'])
        for sn in x['sents']:
            if sn['t1'] in d: ys.append(sn['y']); ss.append(SC[m][d[sn['t1']]])
    return roc_auc_score(ys, ss)

def per_resp(items, sc, thr):
    d, l = [], []
    for x in items:
        if x['onset'] is None: continue
        first = next((t for t, s in series(x, sc) if s >= thr), None)
        ok = first is not None and first >= x['onset']
        d.append(float(ok)); l.append(first - x['onset'] if ok else np.nan)
    return np.array(d), np.array(l)

def boot(a, b, n=1000, seed=0):
    da, la = per_resp(DEV, a, calibrate(CAL, a, 100)); db, lb = per_resp(DEV, b, calibrate(CAL, b, 100))
    rng = np.random.default_rng(seed); N = len(da); rd = []
    for _ in range(n):
        i = rng.integers(0, N, N); rd.append(da[i].mean() - db[i].mean())
    lo, hi = np.percentile(rd, [2.5, 97.5])
    return {'A': a, 'B': b, 'recall_A': da.mean(), 'recall_B': db.mean(), 'diff': da.mean() - db.mean(), 'lo': lo, 'hi': hi}

# 기준선 확인 (R0 dev-L과 같아야 함)
for sc, ref in [('B|end', 0.5007), ('B|4', 0.4809), ('E2|4', 0.4506)]:
    got = evaluate(DEV, sc, calibrate(CAL, sc, 100))['recall']
    print(f'{sc:6} R0 {ref:.4f} | 지금 {got:.4f} | {"일치" if abs(got-ref) <= 0.002 else "불일치 — 확인 필요"}')

## 7. 학습 — R1c base와 같은 루프, 학습 정답만 보정 (끝난 시드는 건너뜀)

In [ ]:
KEEP_WEIGHTS = {42}
from transformers import get_linear_schedule_with_warmup
import torch.nn.functional as F, shutil

BACKBONES = {'mini': ('cross-encoder/nli-MiniLM2-L6-H768', 2e-5),
             'base': ('cross-encoder/nli-deberta-v3-base', 2e-5),
             'large': ('lytang/MiniCheck-DeBERTa-v3-Large', 1e-5)}
BATCH_SEQ, MICRO, N_EVALS = 64, 16, 4
BF16 = DEVICE == 'cuda' and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16
if not BF16: print('⚠️ bf16 미지원 GPU — DeBERTa는 fp16에서 불안정할 수 있음 (A100/L4 권장)')

def enc_ids(tok, idxs):
    return tok([ctx_of(P_KEY[g], P_IDX[g]) for g in idxs], [P_TXT[g] for g in idxs], truncation='only_first',
               max_length=MAX_LEN, padding=True, return_tensors='pt').to(DEVICE)

@torch.no_grad()
def score_queries(tok, model, qs, bs=128):
    model.eval(); out = np.zeros(len(qs), dtype=np.float32)
    order = sorted(range(len(qs)), key=lambda i: len(Q_TEXTS[qs[i]]))
    for i in range(0, len(order), bs):
        b = [qs[j] for j in order[i:i + bs]]
        enc = tok([ctx_of(Q_KEYS[q], EQ_IDX[q]) for q in b], [Q_TEXTS[q] for q in b], truncation='only_first',
                  max_length=MAX_LEN, padding=True, return_tensors='pt').to(DEVICE)
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
            out[order[i:i + bs]] = model(**enc).logits[:, 0].float().cpu().numpy()
    model.train()
    return out

CALSUB_Q = sorted({q for x in CALSUB for _, q in x['pts2']})
def calsub_metric(tok, model):
    v = score_queries(tok, model, CALSUB_Q)
    SC['_cur'] = np.zeros(len(Q_KEYS), dtype=np.float32); SC['_cur'][CALSUB_Q] = v
    r4 = evaluate(CALSUB, '_cur|4', calibrate(CALSUB, '_cur|4', 100))
    r2 = evaluate(CALSUB, '_cur|2', calibrate(CALSUB, '_cur|2', 100))
    d4 = r4['delay_tok_median']
    return float((r4['recall'] + r2['recall']) / 2), float(d4) if d4 == d4 else 1e9

Y_ALL = np.isin(P_CAT, ['win', 'post']).astype(np.float32)          # V0b: 문장 단위 시작 이후 = 1

def load_init(name):
    path, _ = BACKBONES[name]
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(path, num_labels=1, ignore_mismatched_sizes=True).to(DEVICE)
    return tok, model

def train_backbone(name, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tok, model = load_init(name)
    lr = BACKBONES[name][1]
    order = list(range(len(P_TXT))); random.Random(seed).shuffle(order)
    n_steps = max(1, len(order) // BATCH_SEQ)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * n_steps), n_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda' and not BF16))
    best, d0 = calsub_metric(tok, model)
    best_key, best_state, hist = (d0 <= 9, best), copy.deepcopy(model.state_dict()), [(0, best, d0)]
    print(f'  [{name} s{seed}] 시작 cal-sub 재현율 {best:.3f}, 지연 {d0} | {n_steps}단계, 학습률 {lr}')
    eval_at = {int(n_steps * (i + 1) / N_EVALS) for i in range(N_EVALS)}
    model.train(); t0 = time.perf_counter()
    for step in range(1, n_steps + 1):
        batch = order[(step - 1) * BATCH_SEQ: step * BATCH_SEQ]
        opt.zero_grad(); tot = 0.0
        for k in range(0, len(batch), MICRO):
            mb = batch[k:k + MICRO]
            with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
                z = model(**enc_ids(tok, mb)).logits[:, 0].float()
            loss = F.binary_cross_entropy_with_logits(z, torch.tensor(Y_ALL[mb], device=DEVICE), reduction='sum') / len(batch)
            scaler.scale(loss).backward(); tot += loss.item()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sch.step()
        if step in eval_at:
            va, dv = calsub_metric(tok, model); hist.append((step, va, dv))
            if (dv <= 9, va) > best_key: best_key, best, best_state = (dv <= 9, va), va, copy.deepcopy(model.state_dict())
            print(f'  [{name} s{seed}] 단계 {step}/{n_steps} | 손실 {tot:.3f} | cal-sub 재현율 {va:.3f}, 지연 {dv} | {(time.perf_counter()-t0)/60:.1f}분')
    model.load_state_dict(best_state)
    return tok, model, hist

def run(name, seed):
    out = os.path.join(R1_DIR, f'{name}_s{seed}')
    sp = os.path.join(out, 'scores.npy')
    if not os.path.exists(sp):
        has_w = any(os.path.exists(os.path.join(out, f)) for f in ['model.safetensors', 'pytorch_model.bin'])
        if os.path.exists(os.path.join(out, 'config.json')) and not has_w:
            print(f'  [{name} s{seed}] 저장이 끊긴 폴더 → 지우고 다시 학습'); shutil.rmtree(out)
        if has_w:
            tok = AutoTokenizer.from_pretrained(out); model = AutoModelForSequenceClassification.from_pretrained(out).to(DEVICE)
        else:
            tok, model, hist = train_backbone(name, seed)
            os.makedirs(out, exist_ok=True); tok.save_pretrained(out); model.save_pretrained(out)
            pd.DataFrame(hist, columns=['step', 'calsub_recall', 'calsub_delay4']).to_csv(os.path.join(out, 'hist.csv'), index=False)
        t0 = time.perf_counter()
        v = score_queries(tok, model, EVAL_Q)
        full = np.full(len(Q_KEYS), np.nan, dtype=np.float32); full[EVAL_Q] = v
        np.save(sp, full)
        if seed not in KEEP_WEIGHTS:                     # 드라이브 용량: 가중치는 시드 42만 남긴다
            for f in ['model.safetensors', 'pytorch_model.bin']:
                if os.path.exists(os.path.join(out, f)): os.remove(os.path.join(out, f))
        print(f'  [{name} s{seed}] 평가 질의 채점 {time.perf_counter()-t0:.0f}초')
        del model; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
    SC[f'{name}_s{seed}'] = np.load(sp)
    print(f'[{name} s{seed}] 준비 완료')

In [ ]:
SEEDS = [42, 7, 2026]
for seed in SEEDS:
    run('base', seed)                     # 드라이브 r4c/base_s*  (SC 키 'base_s*' = 보정 학습 base)
for s in SEEDS:
    for key, path in [(f'R1cbase_s{s}', os.path.join(R1C_DIR, f'base_s{s}', 'scores.npy')),
                      (f'R1cmini_s{s}', os.path.join(R1C_DIR, f'mini_s{s}', 'scores.npy')),
                      (f'R4bV0b_s{s}', os.path.join(R4B_DIR, f'V0b_s{s}', 'scores.npy')),
                      (f'R1bV0b_s{s}', os.path.join(R1B_DIR, f'V0b_s{s}', 'scores.npy'))]:
        if os.path.exists(path): SC[key] = np.load(path)
need = [f'{f}_s{s}' for f in ['R1cbase', 'R4bV0b'] for s in SEEDS]
assert all(k in SC for k in need), f'비교 점수 없음: {[k for k in need if k not in SC]}'
print('비교 점수 준비 완료')

## 8. dev-L 평가 (원래 정답·보정 정답)와 사전 기준 판정

In [ ]:
KINDS = ['end', '4']
FAMS = {'B': ['B'], 'E2': ['E2']}
for f in ['base', 'R1cbase', 'R1cmini', 'R4bV0b', 'R1bV0b']:
    FAMS[f] = [f'{f}_s{s}' for s in SEEDS if f'{f}_s{s}' in SC]
ALL_DONE = len(FAMS['base']) == 3
rows = []
for lab, (cal_, dev_) in LAB_SETS.items():
    for f, ms in FAMS.items():
        for m in ms:
            ra, sa = resp_auc(dev_, m), sent_auc(dev_, m)
            for k in KINDS:
                sc = f'{m}|{k}'
                r = evaluate(dev_, sc, calibrate(cal_, sc, 100)); r10 = evaluate(dev_, sc, calibrate_flag(cal_, sc, 0.10))
                rows.append({'labels': lab, 'family': f, 'model': m, 'kind': k, 'recall': r['recall'], 'delay_med': r['delay_tok_median'],
                             'pre_onset': r['pre_onset_alarm_rate'], 'svc_recall': r10['recall'], 'resp_auc': ra, 'sent_auc': sa})
res = pd.DataFrame(rows)
agg = res.groupby(['labels', 'family', 'kind'])[['recall', 'delay_med', 'pre_onset', 'svc_recall', 'resp_auc', 'sent_auc']].mean().reset_index()
agg['n'] = res.groupby(['labels', 'family', 'kind']).size().values
pd.set_option('display.width', 250)
print(agg.round(3).to_string(index=False))
fam = lambda f, k, lab, col='recall': float(agg[(agg.labels == lab) & (agg.family == f) & (agg.kind == k)][col].iloc[0])

def per_resp(items, sc, thr):
    return np.array([float((lambda f: f is not None and f >= x['onset'])(next((t for t, s in series(x, sc) if s >= thr), None)))
                     for x in items if x['onset'] is not None])
def boot_pair(a, b, lab, n=1000, seed=0):
    cal_, dev_ = LAB_SETS[lab]
    da, db = per_resp(dev_, a, calibrate(cal_, a, 100)), per_resp(dev_, b, calibrate(cal_, b, 100))
    rng = np.random.default_rng(seed); N = len(da)
    rd = [da[i].mean() - db[i].mean() for i in (rng.integers(0, N, N) for _ in range(n))]
    lo, hi = np.percentile(rd, [2.5, 97.5])
    return {'A': a, 'B': b, 'labels': lab, 'diff': da.mean() - db.mean(), 'lo': lo, 'hi': hi}

K = '4'
g1 = fam('base', K, 'C1') - fam('R4bV0b', K, 'C1')
g2 = fam('base', K, 'C1'); bend = fam('B', 'end', 'C1')
g3 = fam('base', K, 'orig') - fam('R1cbase', K, 'orig')
R4C_1, R4C_2, R4C_3 = g1 >= 0.03, g2 >= bend, g3 >= -0.02
tag = '' if ALL_DONE else '  (시드 3개 미완료 — 보고만)'
print(f"\n=== R4c-1 보정 정답 크기 효과: base {g2:.3f} − R4b V0b {fam('R4bV0b', K, 'C1'):.3f} = {g1:+.3f} (≥ +0.03) → {'충족' if R4C_1 else '미달'}{tag} ===")
print(f"=== R4c-2 목표: base 4토큰 {g2:.3f} vs B 문장 끝 {bend:.3f} (지연 {fam('base', K, 'C1', 'delay_med'):.1f} vs {fam('B', 'end', 'C1', 'delay_med'):.1f}) → {'충족' if R4C_2 else '미달'}{tag} ===")
BT = [boot_pair(f'base_s{s}|{K}', 'B|end', 'C1') for s in SEEDS if f'base_s{s}' in SC]
for b in BT:
    print(f"   {b['A']} vs B|end (C1): {b['diff']:+.3f} [{b['lo']:+.3f}, {b['hi']:+.3f}]")
print(f"=== R4c-3 원래 정답 비열등: base 보정 {fam('base', K, 'orig'):.3f} − R1c base {fam('R1cbase', K, 'orig'):.3f} = {g3:+.3f} (≥ −0.02) → {'충족' if R4C_3 else '미달'}{tag} ===")
print(f"\n(보고) 학습 정답 효과(base): C1 {fam('base', K, 'C1') - fam('R1cbase', K, 'C1'):+.3f} | 참고 MiniLM(R4b): {fam('R4bV0b', K, 'C1') - fam('R1bV0b', K, 'C1'):+.3f}")
print(f"(보고) 서비스 10% 재현율 C1: base {fam('base', K, 'C1', 'svc_recall'):.3f} vs R4b V0b {fam('R4bV0b', K, 'C1', 'svc_recall'):.3f} vs R1c base {fam('R1cbase', K, 'C1', 'svc_recall'):.3f}")
print(f"(보고) 완성 문장 AUROC C1: base {fam('base', K, 'C1', 'sent_auc'):.3f} vs R4b V0b {fam('R4bV0b', K, 'C1', 'sent_auc'):.3f} vs R1c base {fam('R1cbase', K, 'C1', 'sent_auc'):.3f}")

## 9. 과제별 (보고)

In [ ]:
tr = []
for lab, (cal_, dev_) in LAB_SETS.items():
    for tt in TASKS:
        its = [x for x in dev_ if x['task_type'] == tt]
        row = {'labels': lab, 'task': tt, 'n_hall': sum(x['onset'] is not None for x in its), 'B|end': evaluate(its, 'B|end', calibrate(cal_, 'B|end', 100))['recall']}
        for f in ['R1cbase', 'R4bV0b', 'base']:
            row[f'{f}|4'] = np.mean([evaluate(its, f'{m}|4', calibrate(cal_, f'{m}|4', 100))['recall'] for m in FAMS[f]])
        tr.append(row)
task_df = pd.DataFrame(tr)
print(task_df.round(3).to_string(index=False))

## 10. 저장

In [ ]:
import shutil
summ = {'all_seeds': ALL_DONE, 'R4c_1': bool(R4C_1), 'gain_vs_R4bV0b_C1': g1, 'R4c_2': bool(R4C_2), 'base4_C1': g2, 'Bend_C1': bend,
        'boot_vs_Bend_C1': BT, 'R4c_3': bool(R4C_3), 'gain_vs_R1cbase_orig': g3}
json.dump(summ, open('r4c_summary.json', 'w'), default=float, ensure_ascii=False, indent=1)
hists = []
for s in SEEDS:
    p = os.path.join(R1_DIR, f'base_s{s}', 'hist.csv')
    if os.path.exists(p):
        h = pd.read_csv(p); h['run'] = f'base_s{s}'; hists.append(h)
out = {'r4c_devL.csv': res, 'r4c_family.csv': agg, 'r4c_task.csv': task_df, 'r4c_hist.csv': pd.concat(hists) if hists else pd.DataFrame()}
for f, d in out.items():
    d.to_csv(f, index=False); d.to_csv(os.path.join(R1_DIR, f), index=False)
shutil.copy('r4c_summary.json', os.path.join(R1_DIR, 'r4c_summary.json'))
print('저장 완료: 드라이브 selfproject/r4c/')